In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CNN Environment Ready!")

In [2]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Convert images into tensors
transform = transforms.ToTensor()

# Load existing training dataset
train_dataset = datasets.CIFAR10(
    root='../Exercise 01 - MLP/data',
    train=True,
    download=False,
    transform=transform
)

# Load existing testing dataset
test_dataset = datasets.CIFAR10(
    root='../Exercise 01 - MLP/data',
    train=False,
    download=False,
    transform=transform
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))
print("CIFAR-10 Dataset Loaded Successfully!")

In [3]:
from torch.utils.data import random_split

# Ensure reproducible split
generator = torch.Generator().manual_seed(42)

# Split training dataset
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=generator
)

# Testing dataset remains separate
print("Training Images:", len(train_subset))
print("Validation Images:", len(val_subset))
print("Testing Images:", len(test_dataset))

print("Dataset splitting completed successfully!")

In [4]:
from torch.utils.data import DataLoader

# Batch size
BATCH_SIZE = 64

# Training DataLoader
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Validation DataLoader
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Testing DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

print("CNN DataLoaders created successfully!")

In [5]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


# Create CNN model
model = CNN()

print(model)

In [6]:
# Test CNN model output shape

sample_images, sample_labels = next(iter(train_loader))

outputs = model(sample_images)

print("Input Shape:", sample_images.shape)
print("Output Shape:", outputs.shape)
print("Model test completed successfully!")

In [7]:
# Define loss function
criterion = nn.CrossEntropyLoss()

# Define optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss Function:", criterion)
print("Optimizer:", optimizer)
print("CNN Loss and Optimizer configured successfully!")

In [8]:
def train_model(model, train_loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("CNN Training function created successfully!")

In [9]:
def evaluate_model(model, data_loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in data_loader:

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("CNN Validation function created successfully!")

In [10]:
# Number of epochs
EPOCHS = 10

# Store training history
train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

# Start CNN training
for epoch in range(EPOCHS):

    train_loss, train_acc = train_model(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_acc = evaluate_model(
        model, val_loader, criterion
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print("-" * 40)

print("CNN Training Completed!")

In [11]:
# CNN Training and Validation Accuracy

plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')

plt.title('CNN - Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()


# CNN Training and Validation Loss

plt.figure(figsize=(8, 5))

plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')

plt.title('CNN - Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [12]:
plt.figure(figsize=(8,5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.title('CNN - Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('CNN - Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [13]:
# Evaluate CNN on test dataset

test_loss, test_accuracy = evaluate_model(
    model,
    test_loader,
    criterion
)

print("CNN Final Test Results")
print("----------------------")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print("CNN Evaluation Completed!")

In [14]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print(classification_report(
    all_labels,
    all_predictions,
    target_names=classes
))

In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print(classification_report(
    all_labels,
    all_predictions,
    target_names=classes
))

In [16]:
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.title('CNN - Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

In [17]:
import os

# Create models folder
os.makedirs("models", exist_ok=True)

# Save trained CNN model
torch.save(
    model.state_dict(),
    "models/cnn_cifar10.pth"
)

print("CNN model saved successfully!")
print("Saved location: models/cnn_cifar10.pth")

In [18]:
model_path = "models/cnn_cifar10.pth"

if os.path.exists(model_path):
    print("CNN model file exists!")
    print(
        "File size:",
        round(os.path.getsize(model_path) / 1024, 2),
        "KB"
    )
else:
    print("CNN model file not found!")

In [19]:
import os

os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/cnn_cifar10.pth")

print("CNN model saved successfully!")
print("Saved location: models/cnn_cifar10.pth")

In [20]:
import os

model_path = "models/cnn_cifar10.pth"

if os.path.exists(model_path):
    print("CNN model file exists!")
    print("File size:", round(
        os.path.getsize(model_path) / 1024, 2
    ), "KB")
else:
    print("CNN model file not found!")

In [21]:
print("================================")
print("EXERCISE 02 - CNN FINAL SUMMARY")
print("================================")
print("Dataset: CIFAR-10")
print("Training Images:", len(train_subset))
print("Validation Images:", len(val_subset))
print("Testing Images:", len(test_dataset))
print("Model: Convolutional Neural Network")
print("Convolution Layers: 2")
print("Filters: 32, 64")
print("Fully Connected Layer: 256")
print("Output Classes: 10")
print("Optimizer: Adam")
print("Loss Function: CrossEntropyLoss")
print("Epochs:", EPOCHS)
print("--------------------------------")
print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy:.2f}%")
print("--------------------------------")
print("Exercise 02 completed!")